## Questão 4

In [63]:
import networkx as nx
import pandas as pd
import numpy as np

df = pd.read_csv("football_edges_filtered.csv")

G = nx.from_pandas_edgelist(
    df,
    source='source',
    target='target',
    edge_attr='weight',
    create_using=nx.DiGraph()
)

### a)

In [64]:
best_winner = max(G.in_degree())

print(f"Numero de Seleções = {G.number_of_nodes()}")
print(f"Numero de Arestas = {G.number_of_edges()}")
if nx.is_strongly_connected(G):
    print("A rede é fortemente conexa")
elif nx.is_weakly_connected(G):
    print("A rede não é fortemente conexa mas é fracamente conexa")
else:
    print("A rede não é nem fortemente nem fracamente conexa")
print(f"A seleção com mais vitórias é o {best_winner[0]} com {best_winner[1]} vitórias")


Numero de Seleções = 237
Numero de Arestas = 2036
A rede não é nem fortemente nem fracamente conexa
A seleção com mais vitórias é o Zimbabwe com 11 vitórias


### b)

In [ ]:
A = nx.adjacency_matrix(G).todense()
lambda_max = max(np.linalg.eigvals(A).real)
converging_alpha = 0.9 / lambda_max

eg_centrality = nx.eigenvector_centrality(G)
katz_centrality = nx.katz_centrality(G, alpha=converging_alpha)
page_rank = nx.pagerank(G)

def get_top_10(centrality):
    return sorted(centrality.items(), key=lambda x: x[1], reverse=True)[:10]

eg10 = get_top_10(eg_centrality)
katz10 = get_top_10(katz_centrality)
pr10 = get_top_10(page_rank)

top10_table = pd.DataFrame({
    "Eigenvector (node)": [n for n, v in eg10],
#    "Eigenvector (value)": [v for n, v in eg10],
    "Katz (node)": [n for n, v in katz10],
#    "Katz (value)": [v for n, v in katz10],
    "PageRank (node)": [n for n, v in pr10],
#    "PageRank (value)": [v for n, v in pr10],
})

pd.set_option("display.width", 200)

print(top10_table)

  Eigenvector (node) Katz (node) PageRank (node)
0            Ecuador     Morocco           Spain
1           Colombia      Canada         England
2          Argentina      Mexico          France
3            Uruguay       Spain       Argentina
4              Spain   Argentina        Paraguay
5             Mexico     Ecuador        Portugal
6           Paraguay     England         Ecuador
7             Brazil    Colombia          Brazil
8             Canada       Egypt     Netherlands
9        Switzerland     Uruguay         Morocco


Os 3 metodos geram um ranking razoavelmente parecido, seleções como Espanha, Equador e Argentina aparecem nos três rankings, além de várias seleções que aparecem em 2 dos 3 rankings, como Inglaterra, Brasil, Canada, Colombia, Paraguai, etc.

Katz e PageRank se mostram bem parecido, com 5 seleções em comum, o que mais muito sentido visto que """o PageRank é basicamente a versão normalizada do Katz""" entre várias aspas

### c)

In [66]:
hubs, authorities = nx.hits(G)

hubs10, auth10 = get_top_10(hubs), get_top_10(authorities)

hits_table = pd.DataFrame({
    "Hub (node):": [n for n, v in hubs10],
#    "Hub (value):": [v for n, v in hubs10],
    "Authority (node)": [n for n, v in auth10],
#    "Authority (value)": [v for n, v in auth10]
    })

print(hits_table)

    Hub (node): Authority (node)
0  Saudi Arabia        Argentina
1       Bolivia          Morocco
2       Uruguay            Spain
3        Brazil         Colombia
4          Peru         Paraguay
5         Chile          Algeria
6        Jordan          Ecuador
7     Venezuela           Brazil
8      Paraguay          Uruguay
9          Iraq           Mexico


Como podemos ver a tabela de Hub não tem nada a ver com os outros índices de centralidade, porém a tabela authority tem, o que faz muito sentido, dado que o número/porcentagem de vitórias contribui muito para a importância de uma seleção.

### d)

#### I)

Antes de fazer o código é razoável imaginar que seleções da américa do sul dominarão o top10 do pagerank visto que a quantidade de partidas do Brasil contra seleções da américa do sul é muito maior do que a quantidade de partidas disputadas contra seleções de qualquer outro continente

In [67]:
brazil_personalization = {n: 0 for n in G.nodes()}
brazil_personalization['Brazil'] = 1


brazil_pr = nx.pagerank(G, personalization=brazil_personalization)

bpr10 = get_top_10(brazil_pr)

brazil_pr_table = pd.DataFrame({
    "Global (node)": [n for n, v in pr10],
#    "Global (value)": [v for n, v in pr10],
    "Brazil (node)": [n for n, v in bpr10],
#    "Brazil (value)": [v for n, v in bpr10]
})

print(brazil_pr_table)

  Global (node) Brazil (node)
0         Spain        Brazil
1       England      Paraguay
2        France     Argentina
3     Argentina       Ecuador
4      Paraguay         Spain
5      Portugal       Bolivia
6       Ecuador        France
7        Brazil         Japan
8   Netherlands        Norway
9       Morocco       Uruguay


Como era de se esperar, as seleções da américa do sul aparecem com mais frequência no page_rank do Brasil, o que faz muito sentido pois existem mais jogos do Brasil contra seleções do mesmo continente.

Além disso pode se notar a presença de Japão e Noruega, que foram justamente as disputas no mata-mata da copa de 2026. Se esse dataset for de partidas recentes isso faria muito sentido, pois provavelmente são partidas com um peso muito alto.

#### II)

In [68]:
conmebol_personalization = {n: 0 for n in G.nodes()}

for country in ["Argentina", "Bolivia", "Brazil", "Chile", "Colombia", "Ecuador", "Paraguay", "Peru", "Uruguay", "Venezuela"]:
    conmebol_personalization[country] += 0.1


conmebol_pr = nx.pagerank(G, personalization=conmebol_personalization)

cpr10 = get_top_10(conmebol_pr)

conmebol_pr_table = pd.DataFrame({
    "Global (node)": [n for n, v in pr10],
#    "Global (value)": [v for n, v in pr10],
    "Conmebol (node)": [n for n, v in cpr10],
#    "Conmebol (value)": [v for n, v in cpr10]
})

print(conmebol_pr_table)

  Global (node) Conmebol (node)
0         Spain        Paraguay
1       England         Ecuador
2        France       Argentina
3     Argentina          Brazil
4      Paraguay        Colombia
5      Portugal         Uruguay
6       Ecuador         Bolivia
7        Brazil       Venezuela
8   Netherlands           Spain
9       Morocco            Peru


Como podemos ver, praticamente só aparecem seleções membras da CONMEBOL exceto pelo Chile que foi substituido pela Espanha. O que era de se esperar, visto que membros de uma confederação jogam muito mais entre si.

In [69]:
q4_pr_table = pd.DataFrame({
    "Global (node)": [n for n, v in pr10],
#    "Global (value)": [v for n, v in pr10],
    "Brazil (node)": [n for n, v in bpr10],
#    "Brazil (value)": [v for n, v in bpr10],
    "Conmebol (node)": [n for n, v in cpr10],
#    "Conmebol (value)": [v for n, v in cpr10]
})

print(q4_pr_table)

  Global (node) Brazil (node) Conmebol (node)
0         Spain        Brazil        Paraguay
1       England      Paraguay         Ecuador
2        France     Argentina       Argentina
3     Argentina       Ecuador          Brazil
4      Paraguay         Spain        Colombia
5      Portugal       Bolivia         Uruguay
6       Ecuador        France         Bolivia
7        Brazil         Japan       Venezuela
8   Netherlands        Norway           Spain
9       Morocco       Uruguay            Peru


O ranking da CONMEBOL se aproxima muito mais do ranking centrado no Brasil do que do ranking global, como ja foi explicado anteriormente.

Para que o ranking da conmebol se aproximasse do ranking global seria necessário aumentar os valores de outros países de diferentes continentes no dicionário de personalização.